# MediCheck-GPT: run the full flow on Colab
Runtime → Change runtime type → **T4 GPU**. Run top to bottom. Every number in the README comes from this notebook's output.

## 0. Setup: GPU, code, data

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # confirm the T4 is attached

In [ ]:
from google.colab import drive
drive.mount('/content/drive')                     # the raw CMS file lives in my Drive
ROOT = '/content/drive/MyDrive/build-transformer'   # same folder as 01_data.ipynb

In [ ]:
!git clone https://github.com/cece15/medicheck-gpt.git /content/medicheck-gpt   # private repo: Colab will ask for a GitHub token
%cd /content/medicheck-gpt
!pip install -q -r requirements.txt

## 1. Data: CSV → one sentence per row → split by hospital

In [ ]:
!python gpt.py --input "$ROOT/data/raw/MUP_INP_RY26_P03_V10_DY24_PrvSvc.CSV" --prepare data/processed
!head -c 600 data/processed/train.txt

## 2. Baseline: bigram (one character of memory)

In [ ]:
!python gpt.py --input data/processed --train checkpoints/bigram.pth --model bigram --context_size 8 --lr 1e-2 --steps 3000

## 3. Transformer: 4 layers, 4 heads, 128-dim, context 271 (one full record)
Cosine LR with warmup, weight decay, early stopping. First run (3 Nov) overfit after ~1,500 steps, so these guards matter.

In [ ]:
!python gpt.py --input data/processed --train checkpoints/gpt.pth --steps 10000 --eval_every 500 --patience 4

In [ ]:
# Loss curve for the slides
import json, matplotlib.pyplot as plt
h = json.load(open('checkpoints/gpt_history.json'))
plt.plot([r['step'] for r in h], [r['train'] for r in h], label='train')
plt.plot([r['step'] for r in h], [r['val'] for r in h], label='validation (unseen hospitals)')
plt.xlabel('step'); plt.ylabel('cross-entropy loss'); plt.legend(); plt.title('MediCheck-GPT training')
plt.savefig('checkpoints/loss_curve.png', dpi=200, bbox_inches='tight'); plt.show()

## 4. Generate

In [ ]:
!python gpt.py --eval checkpoints/gpt.pth --n 3
!python gpt.py --eval checkpoints/gpt.pth --n 2 --prompt "Trinity Health Livonia Hospital in Livonia, MI billed DRG "

## 5. Evaluate: perplexity, token accuracy, ROUGE-L (+ BLEU, BERTScore), record validity

In [ ]:
!python gpt.py --input data/processed --evaluate checkpoints/bigram.pth
!python gpt.py --input data/processed --evaluate checkpoints/gpt.pth --hf_metrics

## 6. Flag silent errors: plant four error types in unseen hospitals' records, measure ROC AUC

In [ ]:
!python gpt.py --input data/processed --flag checkpoints/bigram.pth
!python gpt.py --input data/processed --flag checkpoints/gpt.pth

## 7. Save results to Drive (checkpoints are too big for GitHub)

In [ ]:
!mkdir -p "$ROOT/results" && cp checkpoints/*.json checkpoints/*.png checkpoints/gpt.pth "$ROOT/results/" && ls "$ROOT/results"

## Live demo (Tuesday)
`!python gpt.py --eval checkpoints/gpt.pth --interactive`